In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from collections import Counter
import getpass
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import ImageFile
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models

from google.colab import drive

ImageFile.LOAD_TRUNCATED_IMAGES = True

NOTEBOOK_NAME = "Klasyfikacja_Podpisow_V4_Binarne"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

DRIVE_ARCHIVE_PATH = "/content/drive/MyDrive/Unikalne_JPG.zip"
DRIVE_CSV_PATH = "/content/drive/MyDrive/podpisy_maks_karol.csv"
WORK_DIR = "/content/work"
LOCAL_ARCHIVE_PATH = f"{WORK_DIR}/archive.zip"
EXTRACT_DIR = f"{WORK_DIR}/data"
OUTPUT_DIR = f"{WORK_DIR}/data_split"
MODEL_PATH = f"{WORK_DIR}/{NOTEBOOK_NAME}_{TIMESTAMP}_best_model.pt"
REPORT_PATH = f"{WORK_DIR}/{NOTEBOOK_NAME}_{TIMESTAMP}_test_report.txt"

IMAGE_DIR_CANDIDATES = ["all_images", "images", "imgs", "."]

FILENAME_COL = "filename"
LABEL_COL = "label"

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42

BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 7
LR = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 2

CLASS_MAP = {
    "czytelny": "czytelny",
    "czytelne": "czytelny",
    "nieczytelny": "nieczytelny_brak",
    "nieczytelne": "nieczytelny_brak",
    "brak podpisu": "nieczytelny_brak",
    "brak podp": "nieczytelny_brak",
    "brakpodpisu": "nieczytelny_brak",
    "brak_podpisu": "nieczytelny_brak",
}

def normalize_label(label):
    label = str(label).strip().lower().replace(";", "")
    if label in CLASS_MAP:
        return CLASS_MAP[label]
    raise ValueError(f"Nieznana etykieta: '{label}'")

def clean_columns(df):
    df.columns = [str(c).strip().replace(';', '') for c in df.columns]
    return df

def parse_csv(csv_path):
    encodings = ['utf-8', 'cp1250', 'iso-8859-2', 'latin1']
    separators = [',', ';']

    for enc in encodings:
        for sep in separators:
            try:
                df = pd.read_csv(csv_path, sep=sep, encoding=enc, engine='python')
                df = clean_columns(df)
                if FILENAME_COL in df.columns and LABEL_COL in df.columns:
                    print(f"Poprawnie wczytano plik CSV (kodowanie: {enc}, separator: '{sep}')")
                    return df
            except Exception:
                pass

    for enc in encodings:
        try:
            df = pd.read_csv(csv_path, sep=None, encoding=enc, engine='python')
            df = clean_columns(df)
            return df
        except Exception:
            pass

    raise ValueError(f"Nie udało się odczytać pliku CSV '{csv_path}'. Upewnij się, że nie jest on uszkodzony.")

def find_image_root(root):
    for name in IMAGE_DIR_CANDIDATES:
        p = Path(root) / name
        if p.exists() and p.is_dir():
            return p
    return Path(root)

def find_image_file(filename, source_dir):
    source_dir = Path(source_dir)
    filename = str(filename).strip().replace(";", "")

    candidate = source_dir / filename
    if candidate.exists():
        return candidate

    matches = list(source_dir.rglob(filename))
    if matches:
        return matches[0]

    stem = Path(filename).stem.lower()
    for p in source_dir.rglob('*'):
        if p.is_file() and p.stem.lower() == stem:
            return p

    return None

def decrypt_archive():
    os.makedirs(WORK_DIR, exist_ok=True)

    if not Path(DRIVE_ARCHIVE_PATH).exists():
        raise FileNotFoundError(f"Nie znaleziono archiwum: {DRIVE_ARCHIVE_PATH}")

    print(f"Kopiowanie archiwum do lokalnej pamięci podręcznej Colab...")
    shutil.copy2(DRIVE_ARCHIVE_PATH, LOCAL_ARCHIVE_PATH)

    if os.path.exists(EXTRACT_DIR):
        shutil.rmtree(EXTRACT_DIR)
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "p7zip-full"], check=True)

    password = getpass.getpass("Podaj hasło do archiwum ZIP (AES256): ")

    print("Rozpakowywanie...")
    cmd = [
        "7z", "x",
        LOCAL_ARCHIVE_PATH,
        f"-p{password}",
        f"-o{EXTRACT_DIR}",
        "-y"
    ]
    subprocess.run(cmd, check=True)

    print(f"Rozpakowano do: {EXTRACT_DIR}")

def prepare_split_from_csv():
    csv_path = Path(DRIVE_CSV_PATH)
    if not csv_path.exists():
        raise FileNotFoundError(f"Nie znaleziono pliku CSV na Dysku: {csv_path}")

    image_root = find_image_root(EXTRACT_DIR)

    print(f"CSV: {csv_path}")
    print(f"Katalog obrazów: {image_root}")

    df = parse_csv(csv_path)

    if FILENAME_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(
            f"CSV musi zawierać kolumny '{FILENAME_COL}' i '{LABEL_COL}'. "
            f"Znalezione: {list(df.columns)}"
        )

    df[FILENAME_COL] = df[FILENAME_COL].astype(str).str.strip().str.replace(";", "", regex=False)
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.replace(";", "", regex=False)
    df[LABEL_COL] = df[LABEL_COL].apply(normalize_label)

    df = df[[FILENAME_COL, LABEL_COL]].dropna().drop_duplicates()

    print(f"Liczba rekordów po czyszczeniu: {len(df)}")

    df["filepath"] = df[FILENAME_COL].apply(lambda x: find_image_file(x, image_root))

    missing = df["filepath"].isna().sum()
    if missing > 0:
        missing_files = df.loc[df["filepath"].isna(), FILENAME_COL].head(20).tolist()
        raise FileNotFoundError(f"Brakuje {missing} plików obrazów w archiwum. Przykłady: {missing_files}")

    train_df, temp_df = train_test_split(
        df,
        test_size=(1.0 - TRAIN_SIZE),
        stratify=df[LABEL_COL],
        random_state=RANDOM_STATE,
    )

    val_ratio = VAL_SIZE / (VAL_SIZE + TEST_SIZE)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1.0 - val_ratio),
        stratify=temp_df[LABEL_COL],
        random_state=RANDOM_STATE,
    )

    if Path(OUTPUT_DIR).exists():
        shutil.rmtree(OUTPUT_DIR)

    all_classes = sorted(df[LABEL_COL].unique())

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        for cls in all_classes:
            os.makedirs(Path(OUTPUT_DIR) / split_name / cls, exist_ok=True)

        for _, row in split_df.iterrows():
            src = Path(row["filepath"])
            dst = Path(OUTPUT_DIR) / split_name / row[LABEL_COL] / src.name
            shutil.copy2(src, dst)

    print("\nSTATYSTYKI:")
    print("Całość:", Counter(df[LABEL_COL]))
    print("Train:", Counter(train_df[LABEL_COL]))
    print("Val:", Counter(val_df[LABEL_COL]))
    print("Test:", Counter(test_df[LABEL_COL]))

def build_loaders():
    train_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    eval_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    train_ds = datasets.ImageFolder(Path(OUTPUT_DIR) / "train", transform=train_tfms)
    val_ds = datasets.ImageFolder(Path(OUTPUT_DIR) / "val", transform=eval_tfms)
    test_ds = datasets.ImageFolder(Path(OUTPUT_DIR) / "test", transform=eval_tfms)

    train_targets = [y for _, y in train_ds.samples]
    class_counts = Counter(train_targets)
    class_weights = {cls: 1.0 / cnt for cls, cnt in class_counts.items()}
    sample_weights = [class_weights[y] for y in train_targets]

    sampler = WeightedRandomSampler(
        torch.tensor(sample_weights, dtype=torch.double),
        len(sample_weights),
        replacement=True,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    print(f"\nTrain samples: {len(train_ds)}")
    print(f"Val samples: {len(val_ds)}")
    print(f"Test samples: {len(test_ds)}")
    print(f"Klasy: {train_ds.class_to_idx}")

    return train_loader, val_loader, test_loader, train_ds, test_ds

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, acc, macro_f1, all_targets, all_preds

def train_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nDevice: {device}")

    train_loader, val_loader, test_loader, train_ds, test_ds = build_loaders()

    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_f1 = -1.0
    epochs_no_improve = 0

    print("\nSTART TRENINGU")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"train_loss={running_loss / max(1, len(train_loader)):.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_macro_f1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), MODEL_PATH)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping! Brak poprawy przez {PATIENCE} epok.")
            break

    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    test_loss, test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=test_ds.classes, digits=4)

    full_report = (
        f"TEST loss: {test_loss:.4f}\n"
        f"TEST acc: {test_acc:.4f}\n"
        f"TEST macro_f1: {test_f1:.4f}\n\n"
        f"Confusion matrix:\n{cm}\n\n"
        f"Classification report:\n{report}\n"
    )

    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        f.write(full_report)

    print("\nWYNIKI TESTOWE")
    print(full_report)

def save_back_to_drive():
    drive_out = Path("/content/drive/MyDrive/colab_outputs_podpisy")
    drive_out.mkdir(parents=True, exist_ok=True)

    for path in [MODEL_PATH, REPORT_PATH]:
        if Path(path).exists():
            shutil.copy2(path, drive_out / Path(path).name)

    print(f"Zapisano wyniki do: {drive_out}")

def cleanup_temp_files():
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    print("Usunięto tymczasowe dane z /content/work")

if __name__ == "__main__":
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    try:
        decrypt_archive()
        prepare_split_from_csv()
        train_model()
        save_back_to_drive()
    finally:
        cleanup_temp_files()

Mounting Google Drive...
Mounted at /content/drive
Kopiowanie archiwum do lokalnej pamięci podręcznej Colab...
Podaj hasło do archiwum ZIP (AES256): ··········
Rozpakowywanie...
Rozpakowano do: /content/work/data
CSV: /content/drive/MyDrive/podpisy_maks_karol.csv
Katalog obrazów: /content/work/data
Poprawnie wczytano plik CSV (kodowanie: utf-8, separator: ',')
Liczba rekordów po czyszczeniu: 4633

STATYSTYKI:
Całość: Counter({'nieczytelny_brak': 3414, 'czytelny': 1219})
Train: Counter({'nieczytelny_brak': 2390, 'czytelny': 853})
Val: Counter({'nieczytelny_brak': 512, 'czytelny': 183})
Test: Counter({'nieczytelny_brak': 512, 'czytelny': 183})

Device: cuda

Train samples: 3243
Val samples: 695
Test samples: 695
Klasy: {'czytelny': 0, 'nieczytelny_brak': 1}
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 134MB/s]



START TRENINGU
Epoch 1/30 | train_loss=0.4680 | val_loss=0.4719 | val_acc=0.7957 | val_macro_f1=0.7749
Epoch 2/30 | train_loss=0.4253 | val_loss=0.3878 | val_acc=0.8676 | val_macro_f1=0.8334
Epoch 3/30 | train_loss=0.4120 | val_loss=0.4190 | val_acc=0.8360 | val_macro_f1=0.8017
Epoch 4/30 | train_loss=0.3746 | val_loss=0.4303 | val_acc=0.8547 | val_macro_f1=0.8255
Epoch 5/30 | train_loss=0.3640 | val_loss=0.4045 | val_acc=0.8604 | val_macro_f1=0.8290
Epoch 6/30 | train_loss=0.3430 | val_loss=0.5194 | val_acc=0.8043 | val_macro_f1=0.7814
Epoch 7/30 | train_loss=0.3246 | val_loss=0.4408 | val_acc=0.8633 | val_macro_f1=0.8248
Epoch 8/30 | train_loss=0.3128 | val_loss=0.4421 | val_acc=0.8590 | val_macro_f1=0.8208
Epoch 9/30 | train_loss=0.3050 | val_loss=0.4616 | val_acc=0.8460 | val_macro_f1=0.7851

Early stopping! Brak poprawy przez 7 epok.

WYNIKI TESTOWE
TEST loss: 0.4002
TEST acc: 0.8763
TEST macro_f1: 0.8405

Confusion matrix:
[[140  43]
 [ 43 469]]

Classification report:
         

TERAZ CHCĘ ZMIENIĆ THRESHOLD
Odrzuć dokument jako zły TYLKO WTEDY, gdy jesteś co do tego pewien na co najmniej 80%

In [ ]:
# ==========================================
# SYMULATOR BIZNESOWY - ZMIANA PROGU ODRZUCENIA
# ==========================================
import os
import torch
import numpy as np
import torch.nn as nn
from torchvision import models
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Konfiguracja
REJECT_THRESHOLD = 0.80
SAVED_MODEL_PATH = "/content/drive/MyDrive/colab_outputs_podpisy/Klasyfikacja_Podpisow_V4_Binarne_20260407_160115_best_model.pt"

def evaluate_with_threshold(model, loader, device, threshold):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)

            probs = torch.softmax(logits, dim=1)
            preds = (probs[:, 1] > threshold).long()

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return acc, macro_f1, all_targets, all_preds

print(f"--- TESTOWANIE Z PROGIEM ODRZUCENIA: {REJECT_THRESHOLD * 100}% ---")

# 2. Sprawdzenie, czy zdjęcia są w pamięci roboczej. Jeśli nie - odtwarzamy je!
if not os.path.exists("/content/work/data_split/train"):
    print("⚠️ Brak zdjęć testowych w pamięci. Trwa szybkie rozpakowywanie...")
    decrypt_archive()
    prepare_split_from_csv()
else:
    print("✅ Zdjęcia testowe są gotowe.")

# 3. Przygotowanie danych
device = "cuda" if torch.cuda.is_available() else "cpu"
_, _, test_loader, train_ds, test_ds = build_loaders()

# 4. Ładowanie szkieletu i Twojego modelu z Google Drive
print(f"Pobieranie modelu z: {SAVED_MODEL_PATH}")
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

# Wczytujemy model wskaźnikiem na podany plik
model.load_state_dict(torch.load(SAVED_MODEL_PATH, map_location=device))

# 5. Wykonanie testu
print(f"Trwa ocenianie na progu {REJECT_THRESHOLD}...")
test_acc_t, test_f1_t, y_true_t, y_pred_t = evaluate_with_threshold(model, test_loader, device, REJECT_THRESHOLD)

# 6. Wyniki
cm_t = confusion_matrix(y_true_t, y_pred_t)
report_t = classification_report(y_true_t, y_pred_t, target_names=test_ds.classes, digits=4)

print(f"\nTEST acc (Skuteczność ogólna): {test_acc_t:.4f}")
print(f"TEST macro_f1 (Sprawiedliwość): {test_f1_t:.4f}\n")
print(f"Macierz Konfuzji (Confusion matrix):\n{cm_t}\n")
print(f"Szczegółowy raport:\n{report_t}")

--- TESTOWANIE Z PROGIEM ODRZUCENIA: 80.0% ---
⚠️ Brak zdjęć testowych w pamięci. Trwa szybkie rozpakowywanie...
Kopiowanie archiwum do lokalnej pamięci podręcznej Colab...
Podaj hasło do archiwum ZIP (AES256): ··········
Rozpakowywanie...
Rozpakowano do: /content/work/data
CSV: /content/drive/MyDrive/podpisy_maks_karol.csv
Katalog obrazów: /content/work/data
Poprawnie wczytano plik CSV (kodowanie: utf-8, separator: ',')
Liczba rekordów po czyszczeniu: 4633

STATYSTYKI:
Całość: Counter({'nieczytelny_brak': 3414, 'czytelny': 1219})
Train: Counter({'nieczytelny_brak': 2390, 'czytelny': 853})
Val: Counter({'nieczytelny_brak': 512, 'czytelny': 183})
Test: Counter({'nieczytelny_brak': 512, 'czytelny': 183})

Train samples: 3243
Val samples: 695
Test samples: 695
Klasy: {'czytelny': 0, 'nieczytelny_brak': 1}
Pobieranie modelu z: /content/drive/MyDrive/colab_outputs_podpisy/Klasyfikacja_Podpisow_V4_Binarne_20260407_160115_best_model.pt
Trwa ocenianie na progu 0.8...

TEST acc (Skuteczność ogó

In [ ]:
# ==========================================
# SYMULATOR BIZNESOWY - ZMIANA PROGU ODRZUCENIA
# ==========================================
import os
import torch
import numpy as np
import torch.nn as nn
from torchvision import models
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Konfiguracja
REJECT_THRESHOLD = 0.60
SAVED_MODEL_PATH = "/content/drive/MyDrive/colab_outputs_podpisy/Klasyfikacja_Podpisow_V4_Binarne_20260407_160115_best_model.pt"

def evaluate_with_threshold(model, loader, device, threshold):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)

            probs = torch.softmax(logits, dim=1)
            preds = (probs[:, 1] > threshold).long()

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return acc, macro_f1, all_targets, all_preds

print(f"--- TESTOWANIE Z PROGIEM ODRZUCENIA: {REJECT_THRESHOLD * 100}% ---")

# 2. Sprawdzenie, czy zdjęcia są w pamięci roboczej. Jeśli nie - odtwarzamy je!
if not os.path.exists("/content/work/data_split/train"):
    print("⚠️ Brak zdjęć testowych w pamięci. Trwa szybkie rozpakowywanie...")
    decrypt_archive()
    prepare_split_from_csv()
else:
    print("✅ Zdjęcia testowe są gotowe.")

# 3. Przygotowanie danych
device = "cuda" if torch.cuda.is_available() else "cpu"
_, _, test_loader, train_ds, test_ds = build_loaders()

# 4. Ładowanie szkieletu i Twojego modelu z Google Drive
print(f"Pobieranie modelu z: {SAVED_MODEL_PATH}")
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

# Wczytujemy model wskaźnikiem na podany plik
model.load_state_dict(torch.load(SAVED_MODEL_PATH, map_location=device))

# 5. Wykonanie testu
print(f"Trwa ocenianie na progu {REJECT_THRESHOLD}...")
test_acc_t, test_f1_t, y_true_t, y_pred_t = evaluate_with_threshold(model, test_loader, device, REJECT_THRESHOLD)

# 6. Wyniki
cm_t = confusion_matrix(y_true_t, y_pred_t)
report_t = classification_report(y_true_t, y_pred_t, target_names=test_ds.classes, digits=4)

print(f"\nTEST acc (Skuteczność ogólna): {test_acc_t:.4f}")
print(f"TEST macro_f1 (Sprawiedliwość): {test_f1_t:.4f}\n")
print(f"Macierz Konfuzji (Confusion matrix):\n{cm_t}\n")
print(f"Szczegółowy raport:\n{report_t}")

--- TESTOWANIE Z PROGIEM ODRZUCENIA: 60.0% ---
✅ Zdjęcia testowe są gotowe.

Train samples: 3243
Val samples: 695
Test samples: 695
Klasy: {'czytelny': 0, 'nieczytelny_brak': 1}
Pobieranie modelu z: /content/drive/MyDrive/colab_outputs_podpisy/Klasyfikacja_Podpisow_V4_Binarne_20260407_160115_best_model.pt
Trwa ocenianie na progu 0.6...

TEST acc (Skuteczność ogólna): 0.8561
TEST macro_f1 (Sprawiedliwość): 0.8280

Macierz Konfuzji (Confusion matrix):
[[157  26]
 [ 74 438]]

Szczegółowy raport:
                  precision    recall  f1-score   support

        czytelny     0.6797    0.8579    0.7585       183
nieczytelny_brak     0.9440    0.8555    0.8975       512

        accuracy                         0.8561       695
       macro avg     0.8118    0.8567    0.8280       695
    weighted avg     0.8744    0.8561    0.8609       695



[[157  26]
 [ 74 438]]

 czyli odrzuca tylko 26 prawidłowych